
# Linear Algebra with NumPy: Diagonalization and SVD

This short notebook illustrates four standard matrix factorizations in NumPy:

1. **Similarity diagonalization** of a general $3\times 3$ matrix:
   $
   B = P^{-1} A P
   $
2. **Orthogonal diagonalization** of a real symmetric $3\times 3$ matrix:
   $
   \Lambda = Q^{T} C Q
   $
3. **Singular value decomposition (SVD)** of a $2\times 3$ matrix:
   $
   D = U \Sigma V^{T}
   $
4. **Orthogonal diagonalization of**
   $
   E = D^{T}D,
   \qquad
   M = R^{T} E R
   $
   and a demonstration that the eigenvector matrix $R$ agrees with the right singular vector matrix $V$ from the SVD (up to possible sign choices inside degenerate eigenspaces).

Throughout, we will pay close attention to the **ordering** of eigenvalues and singular values, since NumPy functions do not always return them in the same convention.

> This notebook is intended as a compact teaching note. It is suitable for editing and reuse in personal course materials or a personal website.


In [1]:

import numpy as np
np.set_printoptions(precision=6, suppress=True)

def sort_eig_general(evals, evecs, ascending=True):
    '''
    Sort eigenvalues and corresponding eigenvectors for a general eigendecomposition.
    Columns of evecs are reordered to match evals.
    '''
    idx = np.argsort(evals)
    if not ascending:
        idx = idx[::-1]
    return evals[idx], evecs[:, idx]

def fix_signs_by_reference(A, B):
    '''
    Given two matrices whose columns represent vectors, flip column signs of B
    so each column aligns as closely as possible with the corresponding column of A.
    '''
    B2 = B.copy()
    for j in range(min(A.shape[1], B.shape[1])):
        if np.dot(A[:, j], B2[:, j]) < 0:
            B2[:, j] *= -1
    return B2



## 1. Similarity diagonalization of a non-symmetric full-rank $3\times 3$ matrix

We want a matrix $A$ that is:

- real,
- non-symmetric,
- full rank,
- diagonalizable,
- and has real eigenvalues containing both positive and negative values.

A convenient way to build such an example is to choose a diagonal matrix
$
B = \operatorname{diag}(-2,\,1,\,3),
$
and then choose an invertible matrix $P$. If we define
$
A = PBP^{-1},
$
then $A$ will have the same eigenvalues as $B$, but it will generally not be symmetric.
Equivalently,
$
B = P^{-1} A P.
$


In [2]:

# Choose the diagonal form first
B_target = np.diag([-2.0, 1.0, 3.0])

# Choose an invertible matrix P
P = np.array([
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0]
])

# Build a non-symmetric matrix A similar to B_target
A = P @ B_target @ np.linalg.inv(P)

print("A =")
print(A)
print()
print("Is A symmetric?")
print(np.allclose(A, A.T))
print()
print("det(A) =")
print(np.linalg.det(A))


A =
[[-0.5  1.5 -1.5]
 [-1.   2.   1. ]
 [-2.5  2.5  0.5]]

Is A symmetric?
False

det(A) =
-6.000000000000001


In [3]:

# Compute eigenvalues and eigenvectors of A
evals_A, evecs_A = np.linalg.eig(A)

# For this example the eigenvalues are real, but np.linalg.eig may still
# return complex dtype in general. Here we convert tiny imaginary parts away.
evals_A = np.real_if_close(evals_A)
evecs_A = np.real_if_close(evecs_A)

# Sort from small to large
evals_A, evecs_A = sort_eig_general(evals_A, evecs_A, ascending=True)

P_from_eig = evecs_A
B_from_eig = np.linalg.inv(P_from_eig) @ A @ P_from_eig

print("Eigenvalues of A (sorted ascending):")
print(evals_A)
print()
print("P_from_eig =")
print(P_from_eig)
print()
print("P_from_eig^{-1} A P_from_eig =")
print(B_from_eig)


Eigenvalues of A (sorted ascending):
[-2.  1.  3.]

P_from_eig =
[[-0.707107 -0.707107  0.      ]
 [ 0.       -0.707107  0.707107]
 [-0.707107  0.        0.707107]]

P_from_eig^{-1} A P_from_eig =
[[-2.  0.  0.]
 [-0.  1. -0.]
 [ 0.  0.  3.]]



The matrix $P_{\text{from eig}}$ is built from eigenvectors of $A$, placed as columns in the same order as the sorted eigenvalues. Then
$
P_{\text{from eig}}^{-1} A P_{\text{from eig}}
$
is diagonal (up to numerical roundoff).



## 2. Orthogonal diagonalization of a real symmetric $3\times 3$ matrix

Now consider the symmetric matrix
$
C=\begin{pmatrix}
0 & 1 & 0\\
1 & 0 & 0\\
0 & 0 & 0
\end{pmatrix}.
$
Because $C=C^T$, the spectral theorem tells us that it can be orthogonally diagonalized:
$
\Lambda = Q^T C Q,
$
where $Q$ is orthogonal and $\Lambda$ is diagonal.

For this specific matrix, the eigenvalues are
$
-1,\;0,\;1,
$
already giving the requested pattern: one negative, one zero, and one positive eigenvalue.


In [4]:

C = np.array([
    [0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 0.0]
])

evals_C, Q = np.linalg.eigh(C)  # eigh is for real symmetric / Hermitian matrices

Lambda = Q.T @ C @ Q

print("C =")
print(C)
print()
print("Eigenvalues of C from np.linalg.eigh (ascending by default):")
print(evals_C)
print()
print("Q =")
print(Q)
print()
print("Q^T C Q =")
print(Lambda)
print()
print("Is Q orthogonal?")
print(np.allclose(Q.T @ Q, np.eye(3)))


C =
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 0.]]

Eigenvalues of C from np.linalg.eigh (ascending by default):
[-1.  0.  1.]

Q =
[[-0.707107  0.        0.707107]
 [ 0.707107  0.        0.707107]
 [ 0.        1.        0.      ]]

Q^T C Q =
[[-1.  0. -0.]
 [ 0.  0.  0.]
 [ 0.  0.  1.]]

Is Q orthogonal?
True



A useful fact:

- `np.linalg.eigh` is designed for real symmetric matrices.
- It returns eigenvalues in **ascending order**.
- Its eigenvectors are stored as columns of the returned matrix, matched to that same order.

So here the first column of $Q$ corresponds to the eigenvalue $-1$, the second to $0$, and the third to $1$.



## 3. Singular value decomposition of a $2\times 3$ matrix

Let
$
D=\begin{pmatrix}
1 & 0 & 0\\
0 & 2 & 0
\end{pmatrix}.
$
This is already a very transparent example: its nonzero singular values are $1$ and $2$.

NumPy computes the SVD as
$
D = U \Sigma V^T.
$

A practical point: `np.linalg.svd` returns singular values in **descending** order by default.  
Since we want them in **ascending** order, we will reorder the output ourselves.


In [5]:

D = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 2.0, 0.0]
])

U_desc, s_desc, Vt_desc = np.linalg.svd(D, full_matrices=True)

print("D =")
print(D)
print()
print("Singular values from np.linalg.svd (default descending order):")
print(s_desc)
print()
print("U (descending-order convention) =")
print(U_desc)
print()
print("V^T (descending-order convention) =")
print(Vt_desc)


D =
[[1. 0. 0.]
 [0. 2. 0.]]

Singular values from np.linalg.svd (default descending order):
[2. 1.]

U (descending-order convention) =
[[0. 1.]
 [1. 0.]]

V^T (descending-order convention) =
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


In [6]:

# Reorder the nonzero singular values from small to large
idx = np.argsort(s_desc)  # ascending order for the nonzero singular values
s_asc = s_desc[idx]
U_asc = U_desc[:, idx]

# For the full V^T, keep the extra null-space row as well.
# Since Vt_desc is 3x3 here, its first two rows correspond to the nonzero singular values
# in descending order, and the last row spans the null space.
Vt_full_asc = Vt_desc[[1, 0, 2], :]
V_full_asc = Vt_full_asc.T

# Build the rectangular Sigma in ascending-order convention
Sigma_asc = np.zeros((2, 3))
Sigma_asc[0, 0] = s_asc[0]
Sigma_asc[1, 1] = s_asc[1]

print("Singular values reordered ascending:")
print(s_asc)
print()
print("U_asc =")
print(U_asc)
print()
print("Sigma_asc =")
print(Sigma_asc)
print()
print("V_full_asc^T =")
print(Vt_full_asc)
print()
print("Check D = U_asc Sigma_asc V_full_asc^T:")
print(U_asc @ Sigma_asc @ Vt_full_asc)


Singular values reordered ascending:
[1. 2.]

U_asc =
[[1. 0.]
 [0. 1.]]

Sigma_asc =
[[1. 0. 0.]
 [0. 2. 0.]]

V_full_asc^T =
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Check D = U_asc Sigma_asc V_full_asc^T:
[[1. 0. 0.]
 [0. 2. 0.]]



Because singular vectors are only determined up to sign, you may sometimes see the same decomposition with some columns multiplied by $-1$, provided the matching row/column in the partner factor is changed consistently.

Also note that for a rectangular matrix, $\Sigma$ is generally rectangular, not square.



## 4. Diagonalizing $E=D^T D$ and comparing its eigenvectors with the SVD right singular vectors

Define
$
E = D^T D.
$
A standard theorem says that:

- $E$ is real symmetric and positive semidefinite,
- the eigenvalues of $E$ are the squares of the singular values of $D$,
- the eigenvectors of $E$ are the right singular vectors of $D$.

So if
$
D = U\Sigma V^T,
$
then
$
E = D^T D = V \Sigma^T \Sigma V^T.
$

Therefore, if we orthogonally diagonalize $E$ as
$
M = R^T E R,
$
then $R$ should agree with $V$, up to sign choices and possible rotations inside degenerate eigenspaces.


In [7]:

E = D.T @ D

evals_E, R = np.linalg.eigh(E)  # ascending eigenvalues by default
M = R.T @ E @ R

print("E = D^T D =")
print(E)
print()
print("Eigenvalues of E (ascending):")
print(evals_E)
print()
print("R =")
print(R)
print()
print("M = R^T E R =")
print(M)


E = D^T D =
[[1. 0. 0.]
 [0. 4. 0.]
 [0. 0. 0.]]

Eigenvalues of E (ascending):
[0. 1. 4.]

R =
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

M = R^T E R =
[[0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 4.]]


In [8]:

# Compare with the right singular vectors from the SVD.

V_full_desc = Vt_desc.T

# Reorder the columns of V so they correspond to ascending eigenvalues of E = D^T D.
# In this example that order is [0, 1, 4], so the matching columns are [2, 1, 0]
# relative to the default descending-order SVD output.
V_full_asc = V_full_desc[:, [2, 1, 0]]

# Align signs for visual comparison
V_full_asc_aligned = fix_signs_by_reference(R, V_full_asc)

print("V from the SVD, reordered to ascending order of eigenvalues of D^T D:")
print(V_full_asc)
print()
print("After sign alignment:")
print(V_full_asc_aligned)
print()
print("R from eigh(E):")
print(R)
print()
print("Do they agree up to column signs?")
print(np.allclose(R, V_full_asc_aligned))


V from the SVD, reordered to ascending order of eigenvalues of D^T D:
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

After sign alignment:
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

R from eigh(E):
[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

Do they agree up to column signs?
True



In this example, $R$ and $V$ match exactly up to signs after consistent ordering.

More conceptually:

- if $D$ has singular values $\sigma_1,\sigma_2,\dots$,
- then $D^T D$ has eigenvalues $\sigma_1^2,\sigma_2^2,\dots$,
- and the corresponding eigenvectors are exactly the columns of $V$.

That is why the right singular vectors in the SVD are often introduced through the eigendecomposition of $D^T D$.



## Summary

We have seen four closely related decompositions:

1. **Similarity diagonalization**:
   $
   B=P^{-1}AP
   $
   for a diagonalizable matrix $A$.

2. **Orthogonal diagonalization**:
   $
   \Lambda=Q^T C Q
   $
   for a real symmetric matrix $C$.

3. **SVD**:
   $
   D = U\Sigma V^T
   $
   for a rectangular matrix $D$.

4. **Connection between SVD and $D^T D$**:
   $
   E=D^T D = V\Sigma^T\Sigma V^T.
   $

These examples also illustrate an important practical lesson:  
different NumPy routines use different default orderings, so it is often worth sorting eigenvalues or singular values explicitly and then reordering the corresponding vectors in the same way.



---

**Authorship note.**  
This notebook was prepared as a collaboration between the author and ChatGPT.
